# 02 - Training Smoke Run

Build a tiny `ExperimentConfig` and run `run_experiment` end-to-end
against the bundled `small_vector_001` data.

This notebook is **not** a scientific experiment. It is a smoke test
that the existing single-landscape MaskablePPO + Graphab loop produces
the contract artifacts.

In [1]:
from pathlib import Path
pkg_root = Path.cwd().resolve()
if pkg_root.name != 'habconn':
    pkg_root = Path.cwd().parent
data_dir = pkg_root / 'data' / 'examples' / 'small_vector_001'
graphab_jar = pkg_root / 'tools' / 'graphab.jar'
scratch_root = pkg_root / 'tmp' / 'notebooks'
scratch_root.mkdir(parents=True, exist_ok=True)
print('pkg_root      :', pkg_root)
print('data_dir      :', data_dir)
print('graphab_jar   :', graphab_jar)
print('scratch_root  :', scratch_root)

pkg_root      : C:\Users\dev\work\tum\drl-sp\08_pkg\habconn
data_dir      : C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\data\examples\small_vector_001
graphab_jar   : C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tools\graphab.jar
scratch_root  : C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks


## Build the experiment config

In [2]:
from habconn.training.experiment import ExperimentConfig

config = ExperimentConfig(
    run_name='notebook_smoke',
    seed=42,
    data_dir=data_dir,
    graphab_jar=graphab_jar,
    work_root=scratch_root / 'training_runs',
    output_root=scratch_root / 'experiments',
    budget=3,
    k=10,
    total_timesteps=50,
    n_eval_episodes=1,
)
print(config)

ExperimentConfig(run_name='notebook_smoke', seed=42, data_dir=WindowsPath('C:/Users/dev/work/tum/drl-sp/08_pkg/habconn/data/examples/small_vector_001'), graphab_jar=WindowsPath('C:/Users/dev/work/tum/drl-sp/08_pkg/habconn/tools/graphab.jar'), work_root=WindowsPath('C:/Users/dev/work/tum/drl-sp/08_pkg/habconn/tmp/notebooks/training_runs'), output_root=WindowsPath('C:/Users/dev/work/tum/drl-sp/08_pkg/habconn/tmp/notebooks/experiments'), budget=3, k=10, total_timesteps=50, learning_rate=0.0003, n_steps=8, batch_size=4, n_epochs=2, gamma=0.99, n_eval_episodes=1, n_envs=1, checkpoint_freq=16, selection_metric='mean_final_pc', selection_mode='max')


## Run the experiment

Each environment step calls Graphab CLI exact evaluation; expect a few
minutes for 50 timesteps.

In [3]:
from habconn.training.experiment import run_experiment
summary = run_experiment(config)
print('run_dir              :', summary['run_dir'])
print('total_timesteps      :', summary['total_timesteps'])
print('training_episodes    :', summary['n_training_episodes_logged'])
eval_summary = summary['evaluation']
print('eval_mean_return     :', f"{eval_summary['mean_return']:.3e}")
print('eval_mean_final_pc   :', f"{eval_summary['mean_final_pc']:.6e}")
print('eval_mean_delta_pc   :', f"{eval_summary['mean_delta_pc']:.3e}")
print('selected_candidate   :', summary['selected_candidate_id'],
      f"({summary['selected_candidate_type']})")
print('deployment_selected  :', summary['deployment_selected_pu_ids'])

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3        |
|    ep_rew_mean     | 4e-06    |
| time/              |          |
|    fps             | 0        |
|    iterations      | 1        |
|    time_elapsed    | 28       |
|    total_timesteps | 8        |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 3             |
|    ep_rew_mean          | 4.2e-06       |
| time/                   |               |
|    fps                  | 0             |
|    iterations           | 2             |
|    time_elapsed         | 51            |
|    total_timesteps      | 16            |
| train/                  |               |
|    approx_kl            | 4.8920512e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2       

## Artifact paths

These are written under `run_dir` by the experiment contract.

In [4]:
for key in (
    'config_path', 'metadata_path', 'history_path', 'summary_path',
    'comparison_json_path', 'comparison_csv_path',
    'model_selection_path', 'checkpoint_evaluations_path',
    'deployment_summary_path',
    'selected_planning_units_gpkg_path',
    'selected_planning_units_csv_path',
    'observation_schema_path', 'feature_summary_path',
    'deployment_action_trace_json_path',
    'deployment_action_trace_csv_path',
    'model_path', 'best_model_path',
):
    print(f'{key:36s} {summary[key]}')

config_path                          C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\experiments\notebook_smoke\config.json
metadata_path                        C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\experiments\notebook_smoke\metadata.json
history_path                         C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\experiments\notebook_smoke\history.jsonl
summary_path                         C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\experiments\notebook_smoke\baseline_summary.json
comparison_json_path                 C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\experiments\notebook_smoke\evaluation\comparison.json
comparison_csv_path                  C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\experiments\notebook_smoke\evaluation\comparison.csv
model_selection_path                 C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\experiments\notebook_smoke\selection\model_selection.json


Continue with `03_outputs_evaluation_deployment_inspection.ipynb` to
explore the artifacts.